In [1]:
%pip install -q fastapi uvicorn pyngrok nest-asyncio
%pip install -q transformers accelerate torch torchvision sentencepiece

!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,840 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [101 kB]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [355 B]       
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Hit:9 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,119 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]       
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.

In [2]:
!fuser -k 11434/tcp
!pkill -f ollama
!ollama serve > ollama.log 2>&1 &

In [3]:
!ollama pull qwen3-vl:8b-instruct

In [4]:
!ollama list

NAME                    ID              SIZE      MODIFIED               
qwen3-vl:8b-instruct    0533d74300e4    6.1 GB    Less than a second ago    


In [ ]:
import json
from logging import exception
import asyncio
import nest_asyncio
from typing import List, Union, Dict, Any, Optional
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from pyngrok import ngrok
import uvicorn
import httpx

from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer
from threading import Thread
from huggingface_hub import login

nest_asyncio.apply()

class ChatMessage(BaseModel):
  role: str
  content: str
  images: List[Any] | None = None

class LLMPayload(BaseModel):
  model: str
  messages: List[ChatMessage]
  format: Optional[dict[str, Any]] = None
  temperature: float = 0.0
  think: bool = False
  stream: bool = True
  num_predict: int = 0
  num_ctx: int = 0,
  max_tokens: int = 0

app = FastAPI(title="Medical AI Inference Gateway")

class ModelManager:
    def __init__(self):
        self.models = {}

    def get_model(self, model_name):
        try:
            if model_name not in self.models:
                login("")

                tokenizer = AutoTokenizer.from_pretrained(model_name)
                
                model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype="auto")

                self.models[model_name] = {"tokenizer": tokenizer, "model": model}
            return self.models[model_name]
        except Exception as e:
           return e

manager = ModelManager()

@app.post("/v1/transformers/chat/completions")
async def transformers_generate(payload: LLMPayload):
   model_info = manager.get_model(payload.model)

   tokenizer = model_info["tokenizer"]
   model = model_info["model"]

   inputs = tokenizer.apply_chat_template(
      payload.messages,
      tokenize=True,
      add_generation_prompt=True,
      return_tensors="pt"
   ).to(model.device)

   print(model.config._attn_implementation)
   print(model.device)

   streamer = TextIteratorStreamer(
      tokenizer,
      skip_prompt=True,
      skip_special_tokens=True
   )

   thread = Thread(
      target=model.generate,
      kwargs={
         **inputs,
         "streamer": streamer,
         "max_new_tokens": payload.max_tokens
      }
   )

   def generate():
        thread.start()
        for text in streamer:
            yield text

   return StreamingResponse(generate(), media_type="text/plain")

#    outputs = model.generate(
#       **inputs,
#       max_new_tokens=payload.max_tokens
#    )

#    input_length = inputs["input_ids"].shape[-1]
#    generated_ids = outputs[0][input_length:]

#    response = tokenizer.decode(
#     generated_ids,
#     skip_special_tokens=True
#     )

#    return PlainTextResponse(response.strip())
   

OLLAMA_API_URL = "http://127.0.0.1:11434/v1/chat/completions"

timeout_config = httpx.Timeout(300.0, connect=60.0, read=300.0, write=60.0)
limits_config = httpx.Limits(max_connections=50, max_keepalive_connections=10)

@app.post("/v1/ollama/image/completions")
async def process_vlm_request(payload: LLMPayload):
  """의료 메타데이터 및 Base64 이미지 페이로드를 받아 내부 로드된 VLM 모델로 추론을 수행하는 핵심 엔드포인트 입니다."""

  try:
    ollama_payload = {
        "model": payload.model,
        "messages": payload.model_dump()["messages"],
        "temperature": payload.temperature,
        "stream": True,
        "think": payload.think,
        "max_tokens": payload.max_tokens,
        "options": {"num_predict": payload.num_predict, "num_ctx": payload.num_ctx}
    }

    if payload.format is not None:
      ollama_payload["format"] = payload.format

    async def response_generator():
      async with httpx.AsyncClient(timeout=timeout_config, limits=limits_config) as client:
          try:
              async with client.stream("POST", OLLAMA_API_URL, json=ollama_payload) as response:
                  if response.status_code != 200:
                      yield f"data: {json.dumps({'error': 'Ollama 서버 응답 에러'})}\n\n"
                      return

                  async for line in response.aiter_lines():
                      if line:
                        decoded_line = line.strip()
                          
                        if decoded_line.startswith("data:"):
                            data_content = decoded_line[5:].strip()
            
                            if data_content == "[DONE]":
                                break
            
                            try:
                                chunk_json = json.loads(data_content)
                                chunk_text = chunk_json.get("choices", [{}])[0].get("delta", {}).get("content", "")
            
                                if chunk_text:
                                    yield chunk_text
                                    
                            except Exception as e:
                                print(f"[스트림 파싱 예외 발생]: {e}")

          except httpx.RemoteProtocolError as proto_err:
              print(f"[Gateway Error] 내부 Ollama 서버가 연산 중 연결을 강제 종료했습니다 (VRAM 초과 의심): {proto_err}")
              error_json = {
                  "choices": [{
                      "delta": {"content": "\n\n⚠️ **[프록시 게이트웨이 시스템 안내]** 로컬 AI 엔진 서버의 GPU 리소스(VRAM OOM) 부족으로 인해 분석 도중 연결이 유실되었습니다. 컨텍스트 크기를 줄이거나 하드웨어 상태를 점검하세요."}
                  }]
              }
              yield f"data: {json.dumps(error_json)}\n\n"
              yield "data: [DONE]\n\n"
          except Exception as stream_err:
              print(f"[Gateway Stream Exception]: {stream_err}")
              return

    return StreamingResponse(response_generator(), media_type="text/plain")
  except Exception as e:
    raise HTTPException(status_code=500, detail=f"Internal Server Error: {str(e)}")

NGROK_TOKEN = "3EzUV00bhdXtnBDjZ6hMUxdpIsv_7sYr1oc1Kf28T1JJrg6Yz"
ngrok.set_auth_token(NGROK_TOKEN)

public_url = ngrok.connect(8000, domain="overrate-comprised-outfield.ngrok-free.dev")
print(f"\n 외부 프로세스에서 접근 가능한 API 주소-vlm: {public_url.public_url}/v1/ollama/image/completions\n")

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="debug")
server = uvicorn.Server(config)

loop = asyncio.get_event_loop()
loop.create_task(server.serve())

                                                                                                    
 외부 프로세스에서 접근 가능한 API 주소-vlm: https://overrate-comprised-outfield.ngrok-free.dev/v1/ollama/image/completions



<Task pending name='Task-1' coro=<Server.serve() running at /usr/local/lib/python3.12/dist-packages/uvicorn/server.py:76>>

INFO:     Started server process [1219]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

INFO:     116.45.98.216:0 - "POST /v1/transformers/chat/completions HTTP/1.1" 200 OK
INFO:     116.45.98.216:0 - "POST /v1/transformers/chat/completions HTTP/1.1" 200 OK
INFO:     116.45.98.216:0 - "POST /v1/transformers/chat/completions HTTP/1.1" 200 OK
INFO:     116.45.98.216:0 - "POST /v1/transformers/chat/completions HTTP/1.1" 200 OK
INFO:     116.45.98.216:0 - "POST /v1/transformers/chat/completions HTTP/1.1" 200 OK
INFO:     116.45.98.216:0 - "POST /v1/transformers/chat/completions HTTP/1.1" 200 OK
INFO:     116.45.98.216:0 - "POST /v1/transformers/chat/completions HTTP/1.1" 200 OK
INFO:     116.45.98.216:0 - "POST /v1/transformers/chat/completions HTTP/1.1" 200 OK
INFO:     116.45.98.216:0 - "POST /v1/transformers/chat/completions HTTP/1.1" 200 OK
INFO:     116.45.98.216:0 - "POST /v1/transformers/chat/completions HTTP/1.1" 200 OK
INFO:     116.45.98.216:0 - "POST /v1/transformers/chat/completions HTTP/1.1" 200 OK
